# Example climb: save the princess

**Goal**: maximize the probability of a cure while using the least total ingredients

## 1. Notebook setup

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

sys.path.append('..')

from hill_climber import HillClimber

np.random.seed(315)

## 2. Potion model

### 2.1. Data preparation

In [ ]:
df = pd.read_csv('https://gperdrizet.github.io/FSA_devops/assets/data/unit3/Cure_the_princess.csv')

train_df, test_df = train_test_split(df)
train_df.info()    

### 2.2. Model training

In [ ]:
model = GradientBoostingClassifier()
_ = model.fit(train_df.drop(columns=['Cured']), train_df['Cured'])

### 2.3. Model evaluation

In [ ]:
predictions = model.predict(test_df.drop(columns=['Cured']))

ConfusionMatrixDisplay.from_predictions(
    test_df['Cured'],
    predictions,
    normalize='true',
    colorbar=False
)

plt.title('Test set predictions')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.show()

### 2.4. Retrain with complete dataset

In [ ]:
_ = model.fit(df.drop(columns=['Cured']), df['Cured'])

## 3. Find best potion

### 3.1. Starting potion

In [ ]:
# Get max ingredient values from the dataset
max_values = df.drop(columns='Cured').describe().loc['max']
max_values

In [ ]:
# Create a random starting potion where each ingredient is between 0 and its max value
starting_potion = {}

for feature, value in max_values.items():
    starting_potion[feature] = [np.random.uniform(0, max_values.loc[feature])]

starting_potion_df = pd.DataFrame(starting_potion)
starting_potion_df

### 3.2. Hill climbing objective function

In [ ]:
# Get total max value sum for all ingredients. Will be used to calculate a
# penalty term based on total ingredient values in the potion.
total_max_value = max_values.sum()
print(f'Combined max value for all ingredients: {total_max_value}')

In [ ]:
# Get feature names
features = max_values.index.tolist()
print(f'Features: {features}')

In [ ]:
def score_potion(*args):

    global features
    global model
    global total_max_value

    # Stack columns horizontally to create DataFrame
    potion_df = pd.DataFrame(
        np.column_stack(args),
        columns=features
    )

    # Predict cure probability
    cure_probability = model.predict_proba(potion_df)[0, 1]

    # Calculate total ingredient penalty
    ingredient_sum = potion_df.sum(axis=1).values[0]
    penalty = ingredient_sum / total_max_value
    
    # Return metrics dict and objective value
    metrics = {
        'Cure probability': cure_probability,
        'Ingredient penalty': penalty
    }
    
    objective = cure_probability - penalty

    return metrics, objective

### 3.3. Hill climb optimization

In [ ]:
climber = HillClimber(
    data=starting_potion_df,
    objective_func=score_potion,
    step_spread=0.1,
    T_min=0.001,
    T_max=1.0,
    cooling_rate=1e-10,
    max_time=6 * 60,
    mode='maximize',
    n_replicas=5,
    db_enabled=True,
    db_path='../data/princess.db',
    db_step_interval=1000
)

# Create bounds tuple from 0 to max_values
lower_bounds = np.zeros(len(max_values))
upper_bounds = max_values.values

# Override the bounds that were auto-calculated from the single-row data
climber.bounds = (lower_bounds, upper_bounds)

# Run optimization
best_data, history_df = climber.climb()

In [ ]:
best_data

In [ ]:
history_df.head()

In [ ]:
# Collect best data from all replicas
all_best_data = []

for i, replica in enumerate(climber.replicas):
    data_dict = {f'col_{j}': replica['best_data'][0, j] for j in range(replica['best_data'].shape[1])}
    data_dict['replica_id'] = i
    data_dict['best_objective'] = replica['best_objective']
    all_best_data.append(data_dict)

all_replicas_df = pd.DataFrame(all_best_data)
all_replicas_df

In [ ]:
conn = sqlite3.connect('../data/princess.db')

# Check the schema of run_metadata
query = "PRAGMA table_info(run_metadata)"
schema = pd.read_sql_query(query, conn)
print("run_metadata schema:")
print(schema)

# Get run metadata without ordering by id
query = "SELECT * FROM run_metadata LIMIT 1"
run_meta = pd.read_sql_query(query, conn)
print("\nRun metadata:")
print(run_meta)

# Check replica_status (current state)
query = "SELECT * FROM replica_status ORDER BY replica_id"
replica_status = pd.read_sql_query(query, conn)
print("\nReplica status (current state):")
print(replica_status[['replica_id', 'step', 'best_objective', 'timestamp']])

# Check the LATEST metrics from metrics_history for each replica
query = """
    SELECT replica_id, MAX(step) as max_step
    FROM metrics_history 
    WHERE metric_name = 'Best Objective'
    GROUP BY replica_id
    ORDER BY replica_id
"""
max_steps = pd.read_sql_query(query, conn)
print("\nMax step recorded in metrics_history per replica:")
print(max_steps)

# Get the actual best objective values at those max steps
query = """
    SELECT mh.replica_id, mh.step, mh.value as best_objective_in_db
    FROM metrics_history mh
    INNER JOIN (
        SELECT replica_id, MAX(step) as max_step
        FROM metrics_history
        WHERE metric_name = 'Best Objective'
        GROUP BY replica_id
    ) latest ON mh.replica_id = latest.replica_id AND mh.step = latest.max_step
    WHERE mh.metric_name = 'Best Objective'
    ORDER BY mh.replica_id
"""
best_at_max_step = pd.read_sql_query(query, conn)
print("\nBest objective at max recorded step in metrics_history:")
print(best_at_max_step)

conn.close()